This page walks through the data pipeline from start to finish: where the data comes from, how
FDA's shortage history is rebuilt, how shortage records are traced to drugs, how values are
cleaned, how a shortage onset is defined, and what the final analysis dataset contains. Each
step is checked against the data, and every table is recomputed here from the pipeline's
outputs; numbers quoted in the text are read from those tables. Exploratory analysis of the finished dataset is on the Exploration page.

| Pipeline step | Script | Section |
|---|---|---|
| Download bulk sources | `src/acquisition/download_bulk_files.py` | 1 |
| Fetch archived copies of FDA's shortage list | `src/acquisition/fetch_archived_shortages.py` | 2 |
| Stitch the copies into shortage episodes | `src/acquisition/stitch_archived_shortages.py` | 2 |
| Clean episodes: package codes, reasons, statuses, dates | `src/cleaning/clean_shortage_episodes.py` | 4, 5 |
| Define drug units and link episodes to them | `src/cleaning/map_episodes_to_units.py` | 3, 4 |
| Link recalls to drug units | `src/cleaning/map_recalls_to_units.py` | 9 |
| Build the drug × year panel and its outcome | `src/panel/build_unit_year_panel.py` | 6, 7, 8 |
| Add the predictor columns | `src/panel/add_features.py` | 9 |

A few terms are used throughout:

| Term | Meaning |
|---|---|
| **episode** | one package (a product, strength and pack size from one company) on FDA's shortage list. One drug shortage usually produces many episodes. |
| **NDC** | the package code printed on a drug package |
| **unit** (drug) | an active ingredient given by one route, e.g. *cisplatin, injectable*. This is what the study makes predictions for. |
| **capture** | one copy of FDA's shortage list saved by the Internet Archive on a given day |
| **spell** | one continuous stretch of time a drug was in shortage (section 7) |
| **onset** | the start of a spell: a drug going into shortage when it wasn't already in one |

To reproduce it, run the pipeline in the repository README, then re-run this notebook from the
`site/` folder.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent
assert (REPO_ROOT / "pyproject.toml").exists(), "run this notebook from the site/ folder"
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
INTERIM_DATA_DIR = REPO_ROOT / "data" / "interim"
ARCHIVE_DIR = RAW_DATA_DIR / "cder_archive"

sys.path.insert(0, str(REPO_ROOT / "src" / "acquisition"))
sys.path.insert(0, str(REPO_ROOT / "src" / "cleaning"))
import stitch_archived_shortages as stitching  # noqa: E402
import clean_shortage_episodes as episode_cleaning  # noqa: E402
import map_episodes_to_units as unit_mapping  # noqa: E402

pd.set_option("display.max_colwidth", 90)

## 1. Sources

Every source is US federal public data, downloaded in bulk by the scripts in
`src/acquisition/`. The table lists what one row of each source represents, how many rows it
has, the period it covers, its size on disk, and when we downloaded it.

"Source last updated" is the date the provider stamps on the file, where it has one. "Used for"
separates sources the pipeline already uses from those downloaded for features in a later
milestone.

In [2]:
def load_openfda_file(path: Path) -> tuple[str, list[dict]]:
    with open(path) as file:
        contents = json.load(file)
    return contents["meta"]["last_updated"], contents["results"]


def size_mb(paths: list[Path]) -> float:
    return round(sum(path.stat().st_size for path in paths) / 1e6, 1)


def downloaded_on(path: Path) -> str:
    # The acquisition scripts write each file once, so its modification time is the download date.
    return str(pd.Timestamp(path.stat().st_mtime, unit="s").date())


def date_range(dates: pd.Series) -> str:
    dates = dates.dropna()
    return f"{dates.min():%Y-%m} to {dates.max():%Y-%m}"


def years_in_cms_columns(rows: list[dict]) -> str:
    years = sorted({int(column[-4:]) for column in rows[0] if column.startswith("Tot_Spndng_")})
    return f"{years[0]} to {years[-1]}"

In [3]:
sources = []

archive_paths = sorted(ARCHIVE_DIR.glob("*.csv"))
archive_dates = pd.Series([stitching.snapshot_date(path) for path in archive_paths],
                          dtype="datetime64[ns]")
archive_snapshots = {path: stitching.read_snapshot(path) for path in archive_paths}
archive_rows = sum(len(rows) for rows in archive_snapshots.values())
sources.append({
    "source": "Archived FDA shortage list (CSV)", "provider": "FDA via Internet Archive",
    "one row is": "one package in shortage, in one capture", "rows": archive_rows,
    "coverage": date_range(archive_dates), "size_mb": size_mb(archive_paths),
    "downloaded": downloaded_on(archive_paths[-1]), "source_last_updated": "per capture",
    "used for": "shortage history (the outcome)",
})

shortages_path = RAW_DATA_DIR / "drug-shortages-0001-of-0001.json"
updated, rows = load_openfda_file(shortages_path)
posted = pd.to_datetime(pd.Series([row.get("initial_posting_date") for row in rows]),
                        format="%m/%d/%Y", errors="coerce")
sources.append({
    "source": "Drug Shortages (current)", "provider": "openFDA",
    "one row is": "one package on FDA's list today", "rows": len(rows),
    "coverage": date_range(posted) + " (posting dates)", "size_mb": size_mb([shortages_path]),
    "downloaded": downloaded_on(shortages_path), "source_last_updated": updated,
    "used for": "cross-check of the archive",
})

nsde_paths = sorted(RAW_DATA_DIR.glob("other-nsde-*-of-*.json"))
nsde_rows = []
for path in nsde_paths:
    updated, rows = load_openfda_file(path)
    nsde_rows.extend(rows)
marketing_start = pd.to_datetime(pd.Series([row.get("marketing_start_date") for row in nsde_rows]),
                                 format="%Y%m%d", errors="coerce")
sources.append({
    "source": "NSDE (product lifecycle)", "provider": "openFDA",
    "one row is": "one package ever marketed, with start/end dates", "rows": len(nsde_rows),
    "coverage": date_range(marketing_start) + " (marketing starts)", "size_mb": size_mb(nsde_paths),
    "downloaded": downloaded_on(nsde_paths[0]), "source_last_updated": updated,
    "used for": "linking packages to drugs; drugs on the market each year; supply features",
})

ndc_directory_path = RAW_DATA_DIR / "drug-ndc-0001-of-0001.json"
updated, ndc_directory = load_openfda_file(ndc_directory_path)
sources.append({
    "source": "NDC Directory", "provider": "openFDA",
    "one row is": "one product currently listed", "rows": len(ndc_directory),
    "coverage": "current listings only", "size_mb": size_mb([ndc_directory_path]),
    "downloaded": downloaded_on(ndc_directory_path), "source_last_updated": updated,
    "used for": "bias check only (see section 3)",
})

enforcement_path = RAW_DATA_DIR / "drug-enforcement-0001-of-0001.json"
updated, rows = load_openfda_file(enforcement_path)
report_dates = pd.to_datetime(pd.Series([row.get("report_date") for row in rows]),
                              format="%Y%m%d", errors="coerce")
sources.append({
    "source": "Drug Enforcement (recalls)", "provider": "openFDA",
    "one row is": "one recall of one product", "rows": len(rows),
    "coverage": date_range(report_dates), "size_mb": size_mb([enforcement_path]),
    "downloaded": downloaded_on(enforcement_path), "source_last_updated": updated,
    "used for": "recall features",
})

orange_book_path = RAW_DATA_DIR / "products.txt"
orange_book_products = pd.read_csv(orange_book_path, sep="~", dtype=str)
approval_dates = pd.to_datetime(orange_book_products["Approval_Date"], format="%b %d, %Y",
                                errors="coerce")
sources.append({
    "source": "Orange Book", "provider": "FDA",
    "one row is": "one approved product (strength), incl. discontinued",
    "rows": len(orange_book_products),
    "coverage": date_range(approval_dates) + " (approvals)",
    "size_mb": size_mb([orange_book_path]),
    "downloaded": downloaded_on(orange_book_path), "source_last_updated": "monthly release",
    "used for": "defining drug units (ingredient x route); approval features",
})

for name, file_name, row_meaning in [
    ("Medicare Part B Spending", "ptb_spending.json", "one billing code (HCPCS), yearly columns"),
    ("Medicare Part D Spending", "ptd_spending.json", "one drug per manufacturer, yearly columns"),
]:
    path = RAW_DATA_DIR / file_name
    with open(path) as file:
        rows = json.load(file)
    sources.append({
        "source": name, "provider": "CMS",
        "one row is": row_meaning, "rows": len(rows),
        "coverage": years_in_cms_columns(rows), "size_mb": size_mb([path]),
        "downloaded": downloaded_on(path), "source_last_updated": "annual release",
        "used for": "features (later milestone)",
    })

pd.DataFrame(sources).set_index("source")

,provider,one row is,rows,coverage,size_mb,downloaded,source_last_updated,used for
source,,,,,,,,
Archived FDA shortage list (CSV),FDA via Internet Archive,"one package in shortage, in one capture",249893,2019-10 to 2026-10,79.7,2026-10-10,per capture,shortage history (the outcome)
Drug Shortages (current),openFDA,one package on FDA's list today,1582,2012-01 to 2026-10 (posting dates),3.0,2026-10-10,2026-10-09,cross-check of the archive
NSDE (product lifecycle),openFDA,"one package ever marketed, with start/end dates",670116,1892-01 to 2026-10 (marketing starts),260.8,2026-10-10,2026-10-09,linking packages to drugs; drugs on the market each year; supply features
NDC Directory,openFDA,one product currently listed,138339,current listings only,245.4,2026-10-10,2026-10-09,bias check only (see section 3)
Drug Enforcement (recalls),openFDA,one recall of one product,18002,2012-06 to 2026-09,28.5,2026-10-10,2026-10-06,recall features
Orange Book,FDA,"one approved product (strength), incl. discontinued",48761,1982-01 to 2026-09 (approvals),7.5,2026-10-10,monthly release,defining drug units (ingredient x route); approval features
Medicare Part B Spending,CMS,"one billing code (HCPCS), yearly columns",799,2020 to 2024,1.3,2026-10-10,annual release,features (later milestone)
Medicare Part D Spending,CMS,"one drug per manufacturer, yearly columns",14536,2020 to 2024,22.3,2026-10-10,annual release,features (later milestone)


Two things in this table matter for everything that follows. The **archived shortage list is
the only source of shortage history**: FDA's current file holds today's list and drops resolved
shortages after about six months. And **NSDE, not the NDC Directory, is the product catalog we
join to**, because NSDE keeps packages after they leave the market and the NDC Directory does not
(section 3 measures what that difference costs).

A few of NSDE's marketing start dates are implausibly early (as early as 1892), presumably
placeholder values. Only market presence from 2019 on matters here, so they don't affect the
analysis. The Orange Book's coverage starts at 1982 because it dates earlier approvals only as
"Approved Prior to Jan 1, 1982", which this table can't read as a date.

## 2. The shortage archive

FDA's shortage list is a snapshot, so our shortage history is rebuilt from copies of it saved by
the Internet Archive. How often those copies were taken determines what we can see: a shortage
that starts and ends between two copies never appears.

### How often the list was captured

In [4]:
capture_dates = pd.DatetimeIndex(sorted(set(archive_dates)))
captures_by_year = pd.Series(capture_dates.year).value_counts().sort_index()
captures_by_year.rename_axis("year").rename("days with a capture").to_frame().T

year,2019,2020,2021,2022,2023,2024,2025,2026
days with a capture,2,16,10,2,9,20,16,45


In [5]:
gaps = pd.DataFrame({
    "from": capture_dates[:-1],
    "to": capture_dates[1:],
    "gap_days": (capture_dates[1:] - capture_dates[:-1]).days,
})
print(f"captures: {len(capture_dates)} days, "
      f"{capture_dates[0]:%Y-%m-%d} to {capture_dates[-1]:%Y-%m-%d}")
print(f"gap between captures: median {gaps['gap_days'].median():.0f} days, "
      f"longest {gaps['gap_days'].max()} days")

gaps[gaps["gap_days"] > stitching.PURGE_WINDOW_DAYS].reset_index(drop=True)

captures: 120 days, 2019-10-20 to 2026-10-07
gap between captures: median 7 days, longest 308 days


,from,to,gap_days
0,2021-11-30,2022-10-04,308
1,2023-05-14,2023-11-14,184


In [6]:
median_gap_by_year = gaps.groupby(gaps["to"].dt.year)["gap_days"].median()
median_gap_by_year.rename_axis("year").rename("median days between captures").to_frame().T

year,2019,2020,2021,2022,2023,2024,2025,2026
median days between captures,70.0,10.0,35.0,155.0,23.0,10.0,9.0,7.0


**Coverage is uneven.** 2019 and 2022 each have only two capture days, and 2021 to 2023 have
median gaps of three weeks to five months. 2020 and 2024 to 2025 have a capture about every ten
days, and only 2026 is roughly weekly. The overall median gap above is driven by the dense years.

Sparse captures lose less than they might seem to, because FDA keeps a resolved shortage on its
list for about six months. Any capture within six months of a shortage still records it. What
does lose shortages is a gap **longer than that purge window**, and there are two. A shortage that
began, was resolved, and was removed entirely inside one of them is missing from our data.
Shortages that began inside a gap and were still listed at the next capture are kept, and the
panel step uses FDA's posting date for those when the archive confirms it.

### The archive starts in October 2019

The first capture is from October 2019. A shortage posted earlier appears only if it was still
on FDA's list then, so the earlier a posting year, the more it is limited to long-running
shortages. Counting episodes (one package's shortage) by the year FDA posted them shows the
effect: the years before 2019 are only the survivors.

In [7]:
episodes = pd.read_parquet(INTERIM_DATA_DIR / "shortage_episodes_clean.parquet")
print(f"episodes stitched from the archive: {len(episodes):,}")

episodes_by_posting_year = episodes["posted"].dt.year.value_counts().sort_index()
episodes_by_posting_year.loc[2012:].rename_axis("posting year").rename("episodes").to_frame().T

episodes stitched from the archive: 11,265


posting year,2012.0,2013.0,2014.0,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023.0,2024.0,2025.0,2026.0
episodes,397,8,163,77,124,665,1613,1077,1567,1031,1406,1642,444,595,324


This is why the panel's label years start in 2020, the first full year after the first
capture. Years before that would undercount shortages, and undercount short ones most.

## 3. From shortage records to drugs

FDA's list and our study count different things.

- **FDA lists packages.** Each row of the shortage list is one specific package, such as
  *Fresenius Kabi's furosemide 20 mg/2 mL vial*, and we call each one an **episode**. When a drug
  goes short, FDA usually lists many of its packages (every company, strength and pack size), so
  one drug shortage produces many episodes.
- **Our study counts drugs.** A drug, which we call a **unit**, is an active ingredient given by
  one route, such as *furosemide, injectable*. That is the level at which we ask "did this drug go
  into shortage this year?"

So every episode has to be traced back to the drug it is a package of. We call that **linking**.
The shortage record only carries a package code, so linking goes through two FDA catalogs:

| Step | Look up in | to find | Example |
|---|---|---|---|
| 1 | the shortage record's text | the package code (**NDC**) | `63323-280-02` |
| 2 | **NSDE**, FDA's catalog of every package ever sold | the FDA approval (**application number**) the package is sold under | `NDA018902` |
| 3 | the **Orange Book**, FDA's list of approved drugs | that approval's active ingredient and route | furosemide, injection |

### One record, step by step

Here is that example traced through the actual data.

In [8]:
episode_ndcs = pd.read_parquet(INTERIM_DATA_DIR / "shortage_episode_ndcs.parquet")
episode_units = pd.read_parquet(INTERIM_DATA_DIR / "episode_units.parquet")
units = pd.read_parquet(INTERIM_DATA_DIR / "units.parquet")
nsde = unit_mapping.load_nsde()
orange_book = unit_mapping.load_orange_book()
episode_ndcs = unit_mapping.resolve_ambiguous_ndcs(episode_ndcs, set(nsde["ndc11"]))

EXAMPLE_NDC11 = "63323028002"
example_ndc = episode_ndcs[episode_ndcs["ndc11"] == EXAMPLE_NDC11].iloc[0]
example_episode = episodes.loc[example_ndc["episode_id"]]
example_package = nsde[nsde["ndc11"] == EXAMPLE_NDC11].iloc[0]
example_approval = orange_book[orange_book["application"] == example_package["application"]].iloc[0]
example_unit = units.merge(
    episode_units[episode_units["episode_id"] == example_ndc["episode_id"]], on="unit_id").iloc[0]

pd.DataFrame([
    ("shortage record: company", example_episode["company"]),
    ("shortage record: FDA's description", example_episode["presentation"]),
    ("step 1: package code found in the text", example_ndc["ndc_raw"]),
    ("step 1: the same code in NSDE's 11-digit form", example_ndc["ndc11"]),
    ("step 2: NSDE says it is sold under approval", example_package["application"]),
    ("step 3: Orange Book ingredient for that approval", example_approval["ingredient"]),
    ("step 3: Orange Book route for that approval", example_approval["route"]),
    ("result: the drug (unit) this episode belongs to",
     f"{example_unit['ingredient']}, {example_unit['route_group']} "
     f"(unit {example_unit['unit_id']})"),
], columns=["", "value"]).set_index("")

,value
,
shortage record: company,"Fresenius Kabi USA, LLC"
shortage record: FDA's description,"20 mg per 2 mL (10 mg per mL), Amber, single dose vial (NDC 63323-280-02)"
step 1: package code found in the text,63323-280-02
step 1: the same code in NSDE's 11-digit form,63323028002
step 2: NSDE says it is sold under approval,NDA018902
step 3: Orange Book ingredient for that approval,FUROSEMIDE
step 3: Orange Book route for that approval,INJECTION
result: the drug (unit) this episode belongs to,"FUROSEMIDE, injectable (unit 1422)"


### How many episodes make it through each step

Every episode goes through the same three steps. The table follows all of them: each row is the
number still traced after that step, and "lost here" is how many failed at that step. The last
row is the extra check that the approval points to a single drug.

In [9]:
usable_ndcs = episode_ndcs.dropna(subset=["ndc11"])
ndcs_in_nsde = usable_ndcs.merge(nsde, on="ndc11")
orange_book_applications = set(orange_book["application"])
ndcs_in_orange_book = ndcs_in_nsde[ndcs_in_nsde["application"].isin(orange_book_applications)]

funnel = pd.DataFrame({
    "": [
        "all shortage episodes in the archive",
        "step 1: a package code (NDC) found in the text",
        "step 2: that package found in NSDE",
        "step 3: its approval is an Orange Book prescription or discontinued product",
        "linked: assigned to one drug (a few by also matching the drug name)",
    ],
    "episodes still traced": [
        len(episodes),
        usable_ndcs["episode_id"].nunique(),
        ndcs_in_nsde["episode_id"].nunique(),
        ndcs_in_orange_book["episode_id"].nunique(),
        episode_units["episode_id"].nunique(),
    ],
}).set_index("")
share_traced = funnel["episodes still traced"] / len(episodes)
funnel["share of all episodes"] = share_traced.map("{:.1%}".format)
funnel["lost here"] = (-funnel["episodes still traced"].diff()).fillna(0).astype(int)
funnel

,episodes still traced,share of all episodes,lost here
,,,
all shortage episodes in the archive,11265,100.0%,0
step 1: a package code (NDC) found in the text,11217,99.6%,48
step 2: that package found in NSDE,11141,98.9%,76
step 3: its approval is an Orange Book prescription or discontinued product,10661,94.6%,480
linked: assigned to one drug (a few by also matching the drug name),10607,94.2%,54


Most of the loss happens at step 3. Steps 1 and 2 lose very few, because FDA's text nearly
always contains a readable package code and NSDE keeps packages even after they leave the market.

### Why the unlinked episodes didn't link

Each unlinked episode is listed under the first step it failed. For step 3, the label in
parentheses is how NSDE classifies the product.

In [10]:
application_units = orange_book.merge(units, on=["ingredient", "route_group"])[
    ["application", "unit_id", "ingredient"]].drop_duplicates()
episode_applications = (ndcs_in_nsde[["episode_id", "application", "marketing_category"]]
                        .drop_duplicates(["episode_id", "application"]))

PLAIN_REASONS = {
    "no usable NDC": "step 1: no readable package code in the text",
    "NDC not in NSDE": "step 2: package code not in NSDE",
    "several units on application, name didn't decide":
        "linked: approval covers several drugs and the record's name didn't say which",
}


def plain_reason(reason: str) -> str:
    if reason.startswith("not in Orange Book: "):
        product_type = reason.removeprefix("not in Orange Book: ")
        return f"step 3: no Orange Book prescription product for the approval ({product_type})"
    return PLAIN_REASONS[reason]


unlinked = episodes.index.difference(episode_units["episode_id"])
reasons = pd.Series([
    plain_reason(unit_mapping.unmatched_reason(episode_id, episode_ndcs, episode_applications,
                                               orange_book_applications))
    for episode_id in unlinked
])
reasons.value_counts().rename_axis("why the episode didn't link").rename("episodes").to_frame()

,episodes
why the episode didn't link,
step 3: no Orange Book prescription product for the approval (bla),222
step 3: no Orange Book prescription product for the approval (unapproved drug other),126
step 3: no Orange Book prescription product for the approval (unapproved drug for use in drug shortage),81
step 2: package code not in NSDE,76
linked: approval covers several drugs and the record's name didn't say which,54
step 1: no readable package code in the text,48
step 3: no Orange Book prescription product for the approval (anda),28
step 3: no Orange Book prescription product for the approval (nda),9
step 3: no Orange Book prescription product for the approval (otc monograph drug),8


The biggest groups are outside the Orange Book by design. **Biologics** ("bla") are listed in
FDA's Purple Book instead. **Unapproved drugs** include imports FDA allowed *because of* a
shortage, which are a consequence of a shortage rather than a drug at risk of one. So our study
covers **approved, non-biologic prescription drugs**, and we say so wherever we report results.

### Why we link through NSDE and the Orange Book, not the NDC Directory

The obvious alternative is to link episodes through FDA's **NDC Directory**. The directory
lists only products on the market *today*, so a package that has since been discontinued can't be
found in it. The table compares the share of episodes each method can link, separately for
ordinary shortage episodes and for episodes where FDA listed the package as being
discontinued.

In [11]:
live_directory_ndc11s = set()
for product in ndc_directory:
    for package in product.get("packaging") or []:
        segments = tuple(package["package_ndc"].split("-"))
        if len(segments) == 3:
            ndc11, _ = episode_cleaning.to_ndc11(segments)
            if ndc11:
                live_directory_ndc11s.add(ndc11)

ndc_in_live_directory = usable_ndcs["ndc11"].isin(live_directory_ndc11s)
found_in_live_directory = ndc_in_live_directory.groupby(usable_ndcs["episode_id"]).any()
reach = pd.DataFrame({"package code found in the live NDC Directory": found_in_live_directory})
reach = reach.reindex(episodes.index, fill_value=False)
reach["linked via NSDE + Orange Book (ours)"] = reach.index.isin(episode_units["episode_id"])
reach["discontinuation"] = episodes["statuses"].str.contains("To Be Discontinued")

(reach.groupby("discontinuation").mean().map("{:.0%}".format)
      .rename(index={False: "ordinary shortage episodes", True: "discontinuation episodes"})
      .rename_axis(""))

,package code found in the live NDC Directory,linked via NSDE + Orange Book (ours)
,,
ordinary shortage episodes,78%,95%
discontinuation episodes,22%,92%


The live directory finds the package code for most ordinary shortage episodes but only a small
share of discontinuations, because a discontinued product drops out of today's catalog. (Finding
the code there is a weaker test than linking to a drug, so this comparison if anything flatters
the directory.) Using it would have quietly removed most discontinuation episodes, which are
exactly the cases research question 3 is about. Our route links both groups at nearly the same
rate.

## 4. Cleaning: before and after

FDA's shortage list is typed by hand, so the same fact is often written several ways. This
section shows each cleaning step on real values: what FDA wrote, and what the pipeline turned it
into. The code is in `src/cleaning/clean_shortage_episodes.py` and
`src/cleaning/map_episodes_to_units.py`.

### Package codes (NDCs)

FDA writes package codes inside free text, in several layouts. A package code has three parts
(company, product, package), and the same code can be written with 10 digits in three different
layouts. NSDE stores the standard 11-digit form, so every code is converted to that. Codes whose
layout can't be read from the text are checked against NSDE: if exactly one reading exists there,
we use it.

In [12]:
FORMAT_MEANINGS = {
    "standard_10": "10-digit code in a standard layout, converted to 11 digits",
    "standard_11": "already in the 11-digit form",
    "resolved_via_nsde": "layout unclear from the text; exactly one reading exists in NSDE",
    "ambiguous_10": "layout unclear and no single reading in NSDE (not used)",
    "invalid": "typo in FDA's text: wrong number of digits (not used)",
}

with_text = episode_ndcs.join(episodes["presentation"], on="episode_id")
format_examples = with_text.drop_duplicates("ndc_format").set_index("ndc_format")
pd.DataFrame(index=list(FORMAT_MEANINGS), data={
    "what it means": pd.Series(FORMAT_MEANINGS),
    "codes found": episode_ndcs["ndc_format"].value_counts(),
    "example: FDA's text": format_examples["presentation"].str.slice(0, 70),
    "example: code found": format_examples["ndc_raw"],
    "example: 11-digit result": format_examples["ndc11"],
}).rename_axis("format")

,what it means,codes found,example: FDA's text,example: code found,example: 11-digit result
format,,,,,
standard_10,"10-digit code in a standard layout, converted to 11 digits",10707,2mg/mL (NDC 57894-200-01),57894-200-01,57894020001
standard_11,already in the 11-digit form,961,"12.5mg, 30 Tablets (NDC 45802-0103-65)",45802-0103-65,45802010365
resolved_via_nsde,layout unclear from the text; exactly one reading exists in NSDE,66,CARBIDOPA & LEVODOPA ER TABLETS 50MG/200MG 100CT; (NDC 1672907901),1672907901,16729007901
ambiguous_10,layout unclear and no single reading in NSDE (not used),7,0.5 mg 100 count bottle (NDC 0781-83-9101),0781-83-9101,NaN
invalid,typo in FDA's text: wrong number of digits (not used),18,"2g/vial, (NDC 21586-012-2)",21586-012-2,NaN


In [13]:
codes_per_episode = episode_ndcs.groupby("episode_id").size()
print(f"episodes whose text lists more than one package code: {(codes_per_episode > 1).sum()}")

episodes whose text lists more than one package code: 354


### Reasons for shortage

FDA's reason field is free text. The pipeline maps every spelling to one of nine cause
categories (the eight reasons manufacturers report to FDA, plus manufacturing delay). An episode
that lists several reasons gets several categories. Blank values and non-reasons such as
"Available" or "TBD" count as missing. Every mapping is listed in `REASON_MAP` in
`clean_shortage_episodes.py`; a spelling not in that table stops the pipeline, so a new variant
always gets a human decision.

In [14]:
raw_reasons = episodes["reason"].map(episode_cleaning.normalize_reason)
spelling_count = raw_reasons[~raw_reasons.isin(episode_cleaning.NOT_A_REASON)].nunique()
print(f"distinct reason spellings in FDA's text: {episodes['reason'].nunique()} "
      f"({spelling_count} after ignoring capital letters, punctuation and non-reasons)")

spellings_by_category = {}
for spelling, categories in episode_cleaning.REASON_MAP.items():
    for category in categories:
        spellings_by_category.setdefault(category, []).append(spelling)

reason_table = pd.DataFrame({
    "spellings mapped here": {category: len(spellings)
                              for category, spellings in spellings_by_category.items()},
    "example spellings": {category: " / ".join(spellings[:3])
                          for category, spellings in spellings_by_category.items()},
}).loc[episode_cleaning.REASON_CATEGORIES]
reason_table.loc["(missing or not a reason)"] = [
    len(episode_cleaning.NOT_A_REASON),
    " / ".join(repr(value) for value in sorted(episode_cleaning.NOT_A_REASON))]
reason_table.rename_axis("category")

distinct reason spellings in FDA's text: 70 (51 after ignoring capital letters, punctuation and non-reasons)


,spellings mapped here,example spellings
category,,
demand_increase,16,demand increase for the drug / demand increase for drug / demand increase in the drug
active_ingredient,14,shortage of an active ingredient / shortage of active ingredient / shortage of an acti...
inactive_ingredient,3,shortage of an inactive ingredient component / shortage of an inactive ingredient / sh...
gmp_compliance,3,requirements related to complying with good manufacturing practices / requirements rel...
discontinuation,4,discontinuation of the manufacture of the drug / discontinuation of manufacture of the...
shipping_delay,6,delay in shipping of the drug / delay in shipping of drug / delay in shipping of the d...
regulatory_delay,2,"regulatory delay / regulatory delay, shortage of active ingredient"
manufacturing_delay,7,manufacturing delay / manufacturing delays / delay in manufacturing
other,4,other / oither / third-party supplier issue


### Statuses

FDA's status values differ only in capitalization. The stitched file records every status an
episode passed through; cleaning merges the spellings.

In [15]:
stitched = pd.read_csv(INTERIM_DATA_DIR / "shortage_episodes.csv", dtype=str,
                       keep_default_na=False)
pd.DataFrame({
    "before (as FDA wrote it)": stitched["statuses"].str.split("|").explode().value_counts(),
    "after": episodes["statuses"].str.split("|").explode().value_counts(),
}).fillna(0).astype(int).rename_axis("status")

,before (as FDA wrote it),after
status,,
Current,7620,7620
Resolved,2228,2228
To Be Discontinued,1629,3325
To be Discontinued,1696,0


### Posting dates

Two kinds of bad dates appear in FDA's Initial Posting Date. `01/01/2012` is a placeholder FDA
uses for shortages that began before its records start (its website shows them as "Prior to
2012"). And some dates are blank or typos, such as a year of 2108. The table counts rows across all
archive captures, so one episode seen in many captures counts many times. Rows without a usable
date are kept: dropping them would make drugs that stayed in shortage look as if they had left
the list and come back.

In [16]:
posting_date_text = pd.Series([row.get("Initial Posting Date", "")
                               for snapshot in archive_snapshots.values() for row in snapshot])
parsed = posting_date_text.map(stitching.parse_fda_date)
is_placeholder = parsed == stitching.PRIOR_TO_2012_SENTINEL
is_unusable = parsed.isna()

pd.DataFrame({
    "rows in the archive": [(~is_placeholder & ~is_unusable).sum(), is_placeholder.sum(),
                            is_unusable.sum()],
    "examples": ["", "01/01/2012",
                 " / ".join(sorted(set(posting_date_text[is_unusable]) - {""})[:5]) + " / (blank)"],
    "treatment": ["used as is",
                  "kept, flagged left_censored (start date unknown)",
                  "kept with no posting date; the episode starts at its first capture"],
}, index=["valid date", "'Prior to 2012' placeholder", "blank, unreadable or impossible year"])

,rows in the archive,examples,treatment
valid date,244776,,used as is
'Prior to 2012' placeholder,2693,01/01/2012,"kept, flagged left_censored (start date unknown)"
"blank, unreadable or impossible year",2424,12/03/2108 / (blank),kept with no posting date; the episode starts at its first capture


In [17]:
no_posting_date = episodes["posted"].isna()
print(f"episodes with no usable posting date: {no_posting_date.sum()}, "
      f"of which still listed as Current at some point: "
      f"{(no_posting_date & episodes['statuses'].str.contains('Current')).sum()}")

episodes with no usable posting date: 132, of which still listed as Current at some point: 132


### Routes

The Orange Book writes routes in 79 different ways, often listing several (`INTRAMUSCULAR,
INTRAVENOUS`). Taking them literally would split one drug into several units. For example, the
cefuroxime vials under one approval are listed under two different route strings. The pipeline
groups them into ten route groups. A product listed under several routes takes the
highest-priority group; injectable comes first, because a product injected even some of the time
must be made as a sterile injectable.

In [18]:
CEFUROXIME_APPROVAL = "ANDA062591"
(orange_book[orange_book["application"] == CEFUROXIME_APPROVAL]
 [["application", "ingredient", "route", "route_group"]].drop_duplicates().set_index("application"))

,ingredient,route,route_group
application,,,
ANDA062591,CEFUROXIME SODIUM,INJECTION,injectable
ANDA062591,CEFUROXIME SODIUM,"INTRAMUSCULAR, INTRAVENOUS",injectable


In [19]:
routes = orange_book[["route", "route_group"]].drop_duplicates()
pd.DataFrame({
    "Orange Book route spellings": routes["route_group"].value_counts(),
    "units": units["route_group"].value_counts(),
    "example spellings": routes.groupby("route_group")["route"].apply(
        lambda spellings: " / ".join(sorted(spellings)[:3])),
}).sort_values("units", ascending=False).rename_axis("route group")

,Orange Book route spellings,units,example spellings
route group,,,
oral,11,1636,"BUCCAL / BUCCAL, SUBLINGUAL / ENTERAL"
injectable,39,927,EPIDURAL / IM-IV / INFILTRATION
topical,5,204,"IONTOPHORESIS / IONTOPHORESIS, TOPICAL / IONTOPHORESIS, TRANSDERMAL"
ophthalmic,2,150,"OPHTHALMIC / OPHTHALMIC, OTIC"
inhalation,2,82,"INHALATION / INHALATION, ORAL"
other,15,68,DENTAL / ENDOCERVICAL / FOR RX COMPOUNDING
nasal,1,47,NASAL
vaginal,2,27,"TOPICAL, VAGINAL / VAGINAL"
otic,1,21,OTIC


The full list of all 79 route spellings and their groups is printed every time
`map_episodes_to_units.py` runs.

## 5. Missing reasons

About half of all episodes have no usable reason. The table counts episodes by the year the
archive first listed them.

In [20]:
first_seen_year = episodes["first_seen"].dt.year
missing_reasons = pd.DataFrame({
    "episodes": first_seen_year.value_counts(),
    "reason missing": (episodes.groupby(first_seen_year)["reason_missing"].mean()
                       .map("{:.0%}".format)),
}).sort_index().rename_axis("year first listed")
missing_reasons.T

year first listed,2019,2020,2021,2022,2023,2024,2025,2026
episodes,2244,947,920,780,3264,1270,1299,541
reason missing,51%,53%,43%,49%,40%,52%,61%,82%


The reason field is the same column in every capture, so these blanks are FDA's, not a
parsing problem. The share is roughly half in every year and rises at the end of the series.
Counted by FDA's posting year instead, the recent years look even worse (most are blank), but
those posting dates are unreliable (section 6). Two consequences:

- Any description of *why* shortages happen can only use the episodes that report a reason, and
  we can't tell whether the missing ones resemble them.
- The reason is reported when FDA posts a shortage, after it has started, so it is **never used
  to predict** a shortage. It is only used to describe them.

## 6. The posting date is often stale

FDA's "Initial Posting Date" looks like the date a package went short, but often it isn't. The
table looks at shortage episodes (not discontinuations) first listed after the archive began.
For each year, it shows how often FDA's posting date was more than a year before the archive
first listed the package, and how far apart the two dates typically were.

In [21]:
is_shortage_episode = episodes["statuses"] != "To Be Discontinued"
after_first_capture = episodes[is_shortage_episode & (episodes["first_seen"] > capture_dates[0])]
years_between = ((after_first_capture["first_seen"] - after_first_capture["posted"]).dt.days
                 / 365.25)
is_stale = years_between >= 1

# FDA can rewrite a row's description, drug name or company name while keeping its posting date
# and package code, and the stitch step then sees a new episode. So an episode is a rewrite if an
# earlier shortage episode had the same drug, company and posting date, or the same package code.
same_listing = ["generic", "company", "posted"]
first_listed_for_same_listing = (episodes[is_shortage_episode].groupby(same_listing)["first_seen"]
                                 .transform("min"))
shortage_ndcs = episode_ndcs.dropna(subset=["ndc11"]).join(
    episodes.loc[is_shortage_episode, "first_seen"], on="episode_id", how="inner")
first_listed_for_ndc = shortage_ndcs.groupby("ndc11")["first_seen"].transform("min")
shares_earlier_package = set(
    shortage_ndcs.loc[shortage_ndcs["first_seen"] > first_listed_for_ndc, "episode_id"])
is_rewrite = ((after_first_capture["first_seen"]
               > first_listed_for_same_listing.loc[after_first_capture.index])
              | after_first_capture.index.isin(shares_earlier_package))

year_first_listed = after_first_capture["first_seen"].dt.year
staleness = pd.DataFrame({
    "shortage episodes first listed": year_first_listed.value_counts(),
    "posting date 1+ year earlier": is_stale.groupby(year_first_listed).mean().map("{:.0%}".format),
    "median years between": years_between.groupby(year_first_listed).median().round(1),
    "of the stale ones: rewrite of an earlier row": is_rewrite[is_stale].groupby(
        year_first_listed[is_stale]).mean().map("{:.0%}".format),
}).sort_index().rename_axis("year first listed")
staleness.T

year first listed,2019,2020,2021,2022,2023,2024,2025,2026
shortage episodes first listed,76,662,684,543,2705,989,706,212
posting date 1+ year earlier,32%,16%,61%,24%,65%,78%,91%,93%
median years between,0.6,0.2,1.2,0.8,1.9,2.7,3.1,3.7
of the stale ones: rewrite of an earlier row,92%,81%,92%,77%,94%,92%,95%,85%


From 2023 on, most newly listed shortage episodes carry a posting date more than a year old,
typically several years. Gaps between captures can't explain differences that large, since the
longest gap is under a year.

The last row shows the main cause: **FDA rewriting a row that was already listed.** When FDA edits
a package's description, drug name or company name (for example "Gland Pharma" becoming "Gland
Pharma Limited"), the stitch step sees a new episode, but FDA keeps the original posting date.
Such a rewrite shares its package code, or its drug, company and posting date, with an earlier
episode. Rewrites account for most stale dates in every year. The rest are packages appearing for
the first time under a drug's existing entry and date.

Either way, the posting date can't mark the start of a shortage on its own. The panel uses it
only when the archive confirms it, meaning the posting date falls between the last capture that
did not list the package and the first that did, and otherwise uses the date the archive first
listed the package. Rewrites belong to a drug already in shortage, so the washout in section 7
merges them into that drug's existing spell.

## 7. Defining a shortage onset

The study predicts **onsets**: a drug going into shortage when it wasn't already in one. Turning
episodes into onsets takes two steps.

1. **Merge a drug's episodes into spells.** A **spell** is one continuous stretch of time a drug
   was in shortage. Each episode covers the time from its start (above) to its resolution, or to
   the last capture listing it. A drug's overlapping episodes form one spell.
2. **Apply a washout.** If FDA edits a package's description, the stitch step sees the edited row
   as a new episode, with a short gap before it. A long gap between captures has the same effect.
   Both would look like the drug leaving shortage and immediately re-entering it. So a gap
   shorter than the **washout period** doesn't end a spell. A drug has to be out of shortage for
   at least that long before a new onset counts.

Episodes only ever listed as "To Be Discontinued" are left out. They record a manufacturer
leaving the market, which is something we use to *predict* shortages (research question 3), not
a shortage itself.

### Choosing the washout period

The table counts onsets per year under different washout periods. "Repeat onsets" is the share
of onsets that come after an earlier spell for the same drug, including a spell already running
when the archive began. That should be rare if a spell really is one shortage.

In [22]:
sys.path.insert(0, str(REPO_ROOT / "src" / "panel"))
import build_unit_year_panel as panel_building  # noqa: E402

intervals = panel_building.shortage_episode_intervals(episodes, episode_units, capture_dates)

washout_rows = {}
for washout_days in [0, 90, 180, 365]:
    spells = panel_building.merge_into_spells(intervals, capture_dates[-1], washout_days)
    known_start = spells[~spells["start_unknown"]]
    onsets_by_year = known_start["start"].dt.year.value_counts()
    row = {year: onsets_by_year.get(year, 0) for year in panel_building.LABEL_YEARS}
    has_earlier_spell = spells.sort_values("start").groupby("unit_id").cumcount() > 0
    row["repeat onsets"] = f"{has_earlier_spell[known_start.index].mean():.0%}"
    washout_rows[f"{washout_days} days"] = row
pd.DataFrame(washout_rows).T.rename_axis("washout")

,2020,2021,2022,2023,2024,2025,repeat onsets
washout,,,,,,,
0 days,45,54,45,147,26,8,56%
90 days,45,36,45,135,12,3,49%
180 days,45,36,45,127,12,3,47%
365 days,44,35,36,33,11,3,14%


With no washout or a short one, about half of onsets are repeats, and 2023 jumps. That
is the 2023 capture gap and FDA's rewrites creating false re-onsets. At 365 days the share of
repeats falls to a small fraction and the yearly counts are stable, so the panel uses
**365 days**. The table counts every spell; the panel's own onsets, below, drop the few for drugs
not yet on the market.

### Checking against FDA's own counts

FDA reports to Congress how many new shortages it identified each year, split by its drug center
(CDER) and its biologics center (CBER). Our data excludes biologics, so the comparable figure is
CDER's. FDA also dates a shortage by its own records rather than by the archive, so the two
should be close but needn't match exactly.

In [23]:
panel = pd.read_parquet(REPO_ROOT / "data" / "processed" / "unit_year_panel.parquet")
our_onsets = panel.groupby("year")["onset"].sum()

# Appendix C of FDA's CY2023-CY2025 Reports to Congress on drug shortages (links below).
FDA_NEW_SHORTAGES = {
    "FDA new shortages, drugs (CDER)": {2023: 33, 2024: 11, 2025: 4},
    "FDA new shortages, biologics (CBER)": {2023: 22, 2024: 4, 2025: 0},
}
pd.DataFrame({
    **FDA_NEW_SHORTAGES,
    "our onsets": {year: our_onsets.get(year, 0) for year in [2023, 2024, 2025]},
}).rename_axis("year").T

year,2023,2024,2025
"FDA new shortages, drugs (CDER)",33,11,4
"FDA new shortages, biologics (CBER)",22,4,0
our onsets,33,11,3


Our onsets match FDA's counts of new drug shortages closely, including the sharp drop after
2023, which FDA and USP both report, so the drop is real and not something our reconstruction
created. This is the strongest single check on the label. FDA's headline totals (55, 15 and 4)
also count biologics, which our data excludes.

FDA counts (Appendix C of each report): [CY2023](https://www.fda.gov/media/179156/download),
[CY2024](https://www.fda.gov/media/189325/download),
[CY2025](https://www.fda.gov/media/193637/download). USP's 2025 summary is covered by
[CIDRAP](https://www.cidrap.umn.edu/resilient-drug-supply/us-drug-shortages-have-declined-now-drag-average-5-years-report-shows).

## 8. The analysis panel

The analysis dataset, `data/processed/unit_year_panel.parquet`, has **one row per drug per year
it was on the market**, for 2020 to 2025. A drug is in year *t* if NSDE shows at least one of its
packages on the market at the end of year *t*−1, so membership uses nothing from year *t*
itself. A package counts as on the market until its marketing end date or, when FDA recorded
none, until NSDE marked it inactive. Drugs launched during a year enter the panel the next year.
Market presence comes through FDA approval numbers, so a drug can count as on the market through
a package under an approval it shares with another drug.

| Column | Meaning | Type |
|---|---|---|
| `unit_id` | which drug (ingredient + route group); names are in `units.parquet` | integer |
| `year` | calendar year, 2020–2025 | integer |
| `in_shortage_on_january_1` | one of the drug's shortage episodes was active on January 1 | true/false |
| `at_risk` | the opposite: the drug *could* newly go short this year | true/false |
| `onset` | **the outcome**: a new shortage spell started this year | 0/1 |

Only `at_risk` rows are used to predict onsets, since a drug already in shortage can't newly go
into one. `at_risk` is decided only from episodes active on January 1, not from later ones. One
consequence of the washout: a drug whose shortage ended within the past year is at risk, but
cannot have a new onset until a full year after that ending. The predictor columns are added in
section 9.

Why 2020 to 2025: the archive starts in October 2019, so spells already running then have
unknown start dates, and 2026 is incomplete.

### Example rows

Cisplatin, a chemotherapy drug with a widely reported 2023 shortage, and atorvastatin (Lipitor),
which was never short. Names are joined in for reading.

In [24]:
panel = pd.read_parquet(REPO_ROOT / "data" / "processed" / "unit_year_panel.parquet")
print(f"panel: {len(panel):,} rows x {panel.shape[1]} columns, "
      f"{panel['unit_id'].nunique():,} drugs")

named_panel = panel.merge(units, on="unit_id")
EXAMPLE_DRUGS = [("CISPLATIN", "injectable"), ("ATORVASTATIN CALCIUM", "oral")]
is_example = pd.Series(list(zip(named_panel["ingredient"], named_panel["route_group"]))).isin(
    EXAMPLE_DRUGS).to_numpy()
panel_examples = named_panel[is_example]
panel_examples[["ingredient", "route_group", *panel.columns]].reset_index(drop=True)

panel: 12,139 rows x 5 columns, 2,223 drugs


,ingredient,route_group,unit_id,year,in_shortage_on_january_1,at_risk,onset
0,ATORVASTATIN CALCIUM,oral,284,2020,False,True,0
1,ATORVASTATIN CALCIUM,oral,284,2021,False,True,0
2,ATORVASTATIN CALCIUM,oral,284,2022,False,True,0
3,ATORVASTATIN CALCIUM,oral,284,2023,False,True,0
4,ATORVASTATIN CALCIUM,oral,284,2024,False,True,0
5,ATORVASTATIN CALCIUM,oral,284,2025,False,True,0
6,CISPLATIN,injectable,731,2020,False,True,0
7,CISPLATIN,injectable,731,2021,False,True,0
8,CISPLATIN,injectable,731,2022,False,True,0
9,CISPLATIN,injectable,731,2023,False,True,1


### Size and class balance

How many rows each year contributes, how many of them are at risk, and how many have the outcome.
This is the class balance a model will face; how the outcome varies is explored on the
Exploration page.

In [25]:
by_year = panel.groupby("year").agg(
    drugs_on_market=("unit_id", "size"),
    at_risk=("at_risk", "sum"),
    onsets=("onset", "sum"),
)
onset_rate = by_year["onsets"] / by_year["at_risk"]
by_year["onset rate among at-risk drugs"] = onset_rate.map("{:.1%}".format)
print(f"total onsets 2020-2025: {by_year['onsets'].sum()}")
by_year.T

total onsets 2020-2025: 161


year,2020,2021,2022,2023,2024,2025
drugs_on_market,1954,1987,2011,2036,2067,2084
at_risk,1858,1878,1911,1911,1950,1992
onsets,44,35,35,33,11,3
onset rate among at-risk drugs,2.4%,1.9%,1.8%,1.7%,0.6%,0.2%


The outcome is rare: about 2% of at-risk rows in most years, and fewer than 1% in 2024 and
2025.

## 9. The full analysis dataset: panel plus features

`data/processed/unit_year_features.parquet` is the panel from section 8 with the predictor
columns added by `src/panel/add_features.py`. It is the dataset the modeling milestone starts
from.

**Timing rule.** For a row (drug, year *t*), every feature is measured as of December 31 of year
*t*−1, or over the calendar year *t*−1. Nothing from year *t* itself is used, so a model only
sees what was knowable before the year it predicts. That is what keeps the features free of the
leakage problem described in the project's background.

### Data dictionary

Built from the file itself: the type column is read from the data, and the cell fails if a
column in the file isn't described here.

In [26]:
features = pd.read_parquet(REPO_ROOT / "data" / "processed" / "unit_year_features.parquet")

DICTIONARY = {
    # column: (meaning, source, measured)
    "unit_id": ("which drug (ingredient + route group)", "Orange Book", "fixed"),
    "year": ("calendar year of the row, 2020-2025", "", "fixed"),
    "in_shortage_on_january_1": ("a shortage episode for the drug was active on January 1",
                                 "shortage archive", "Jan 1 of t"),
    "at_risk": ("could newly go short this year (not in shortage on Jan 1)",
                "shortage archive", "Jan 1 of t"),
    "onset": ("OUTCOME: a new shortage spell started this year", "shortage archive", "during t"),
    "route_group": ("injectable, oral, topical, ... (10 groups)", "Orange Book", "fixed"),
    "active_packages": ("packages on the market", "NSDE", "end of t-1"),
    "active_applications": ("FDA approvals with a package on the market; roughly the number "
                            "of independent manufacturers", "NSDE", "end of t-1"),
    "active_generic_applications": ("of those, generic approvals (ANDAs)", "NSDE", "end of t-1"),
    "active_labelers": ("labeler codes with a package on the market; includes repackagers",
                        "NSDE", "end of t-1"),
    "package_exits_prior_year": ("packages whose marketing ended", "NSDE", "during t-1"),
    "application_exits_prior_year": ("approvals that went from at least one package on the "
                                     "market to none", "NSDE", "during t-1"),
    "discontinuation_notices_prior_year": ("episodes first listed as 'To Be Discontinued'",
                                           "shortage archive", "during t-1"),
    "years_since_first_approval": ("time since the drug's first approval (lower bound for "
                                   "drugs approved before 1982)", "Orange Book", "end of t-1"),
    "cumulative_applicants": ("distinct companies ever approved; a measure of how "
                              "commoditized the drug is", "Orange Book", "end of t-1"),
    "had_prior_shortage": ("a shortage spell ended before this year", "shortage archive",
                           "before t"),
    "recalls_prior_year": ("recall records for the drug published by FDA", "FDA enforcement",
                           "during t-1"),
    "serious_recalls_prior_year": ("of those, Class I (most serious)", "FDA enforcement",
                                   "during t-1"),
}
undescribed = set(features.columns) - set(DICTIONARY)
assert not undescribed, f"columns missing from the dictionary: {undescribed}"

print(f"{len(features):,} rows x {features.shape[1]} columns "
      "(one row per drug per year on the market)")
pd.DataFrame(
    [(column, *DICTIONARY[column], str(features[column].dtype)) for column in features.columns],
    columns=["column", "meaning", "source", "measured", "type"],
).set_index("column")

12,139 rows x 18 columns (one row per drug per year on the market)


,meaning,source,measured,type
column,,,,
unit_id,which drug (ingredient + route group),Orange Book,fixed,int64
year,"calendar year of the row, 2020-2025",,fixed,int64
in_shortage_on_january_1,a shortage episode for the drug was active on January 1,shortage archive,Jan 1 of t,bool
at_risk,could newly go short this year (not in shortage on Jan 1),shortage archive,Jan 1 of t,bool
onset,OUTCOME: a new shortage spell started this year,shortage archive,during t,int64
route_group,"injectable, oral, topical, ... (10 groups)",Orange Book,fixed,str
active_packages,packages on the market,NSDE,end of t-1,int64
active_applications,FDA approvals with a package on the market; roughly the number of independent manufact...,NSDE,end of t-1,int64
active_generic_applications,"of those, generic approvals (ANDAs)",NSDE,end of t-1,int64


### Known limits of the features

- `had_prior_shortage` and `discontinuation_notices_prior_year` come from the archive, which
  starts in October 2019, so for 2020 rows they cover only late 2019. They mean less in early
  years than in later ones.
- `active_labelers` counts repackagers, companies that only relabel another firm's drug, so it
  overstates independent suppliers; `active_applications` is the cleaner measure.
- Packages under an approval that covers several drugs count toward each of them.
- A package counts as on the market until its marketing end date or, if FDA recorded none, its
  NSDE inactivation date. NSDE keeps only a package's latest inactivation, so a package that left
  and returned is treated as on the market throughout.
- Exit timing is approximate. Most exits come from inactivation dates, and 90% of those fall on ten
  dates, mostly at the end of January, consistent with FDA inactivating listings that weren't
  renewed the year before. So `package_exits_prior_year` and `application_exits_prior_year` likely
  record many exits about a year late. Whether to count inactivation-based exits is left to be
  tested when modeling; the Exploration page shows the pattern.
- The Orange Book sometimes spells one product's ingredients two ways (for example two
  multivitamin infusions), which makes it two drugs here, so one shortage can count twice. This
  affects a handful of products and is not corrected.
- Drugs approved before 1982 carry a capped, lower-bound `years_since_first_approval`, because the
  Orange Book doesn't record those approval dates.
- Medicare spending and claims are not included yet. Part D links to drugs only by name, and the
  Part B crosswalk covers only current package codes. That is deferred to the modeling milestone.